In [2]:
import pandas as pd
import glob

# Function to merge tracks and tracksMeta files sequentially with id alignment
def merge_tracks_and_meta(tracks_files, tracks_meta_files):
    combined_tracks_data = pd.DataFrame()
    combined_meta_data = pd.DataFrame()

    id_offset = 0  # Initialize ID offset
    frame_offset = 0  # Initialize Frame offset

    for file_index, (track_file, meta_file) in enumerate(zip(tracks_files, tracks_meta_files), start=1):
        print(f"Processing pair: {track_file} and {meta_file}")

        # Load the tracks and tracksMeta data
        tracks_data = pd.read_csv(track_file)
        meta_data = pd.read_csv(meta_file)

        # Offset the `id` column in both files
        tracks_data["id"] += id_offset
        meta_data["id"] += id_offset

        # Offset the `frame` column in tracks.csv
        tracks_data["frame"] += frame_offset

        # Update the related columns in tracks.csv based on the `id_offset`
        related_id_columns = ["precedingId", "followingId", "leftPrecedingId", "leftAlongsideId",
                              "leftFollowingId", "rightPrecedingId", "rightAlongsideId", "rightFollowingId"]

        for col in related_id_columns:
            if col in tracks_data.columns:
                tracks_data[col] = tracks_data[col].apply(lambda x: x + id_offset if x > 0 else x)

        # Update the offsets for the next file
        id_offset = tracks_data["id"].max() + 1
        frame_offset = tracks_data["frame"].max() + 1000  # Increment the frame offset by 1000

        # Append the processed data to the combined DataFrames
        combined_tracks_data = pd.concat([combined_tracks_data, tracks_data], ignore_index=True)
        combined_meta_data = pd.concat([combined_meta_data, meta_data], ignore_index=True)

    return combined_tracks_data, combined_meta_data

# Define the file patterns for tracks and tracksMeta files
tracks_file_pattern = "*_tracks.csv"  # Update this with the correct path
tracks_meta_file_pattern = "*_tracksMeta.csv"  # Update this with the correct path

# List all tracks and tracksMeta files
tracks_files = sorted(glob.glob(tracks_file_pattern))
tracks_meta_files = sorted(glob.glob(tracks_meta_file_pattern))

# Ensure the number of tracks files matches the number of tracksMeta files
if len(tracks_files) != len(tracks_meta_files):
    raise ValueError("The number of tracks files and tracksMeta files must be the same!")

# Merge the tracks and tracksMeta files with proper id and frame alignment
merged_tracks_data, merged_tracks_meta_data = merge_tracks_and_meta(tracks_files, tracks_meta_files)

# Save the merged data to new CSV files
merged_tracks_data.to_csv("merged_tracks.csv", index=False)
merged_tracks_meta_data.to_csv("merged_tracksMeta.csv", index=False)

print("Merging complete. Files saved as 'merged_tracks.csv' and 'merged_tracksMeta.csv'.")


Processing pair: 01_tracks.csv and 01_tracksMeta.csv
Processing pair: 02_tracks.csv and 02_tracksMeta.csv
Processing pair: 03_tracks.csv and 03_tracksMeta.csv
Merging complete. Files saved as 'merged_tracks.csv' and 'merged_tracksMeta.csv'.
